# Olist Project - Phase 5: Dashboard data preparation (for Power BI)
This notebook creates the clean tables that Power BI will load. Put it in the same folder as `eda_orders_master.csv`, `customers_clean.csv` and `orders_clean.csv`, then run the cells top to bottom.

**Output files (4 tables = a small "star schema"):**
- `dashboard_orders.csv` : one row per delivered order (96,470 rows), the main table
- `dashboard_all_orders.csv` : one row per order of any status (99,441 rows), only for the cancellation KPI
- `dim_date.csv` : a calendar table (needed for month-over-month calculations)
- `dim_state.csv` : state code, state name and region (needed for the map)

**Concept box - why these 4 tables?**
Power BI works best with a **fact table** (the events: orders) surrounded by small **dimension tables** (descriptions: dates, states). This is called a **star schema**. It keeps the model simple, fast and easy to explain.

## Section 0: Setup and loading

**What:** Import pandas and numpy.
**Why:** pandas builds the tables, numpy helps with conditions.
**Watch out:** nothing, this is only setup.

In [2]:
import pandas as pd
import numpy as np

**What:** Load the EDA master table, the clean customers and the clean orders.
**Why:** The master table has the delivered orders, customers gives the real customer id, orders gives all statuses.
**Watch out:** the 3 files must be in the same folder as this notebook.

In [3]:
from pathlib import Path

data_path = Path(
    r'C:\Users\Ankur singh patel\OneDrive\Desktop\Brazillian E-Commerce Project\notebook'
)

df = pd.read_csv(
    data_path / 'eda_orders_master.csv',
    parse_dates=[
        'order_purchase_timestamp',
        'order_delivered_customer_date',
        'order_estimated_delivery_date'
    ]
)

customers = pd.read_csv(data_path / 'customers_clean.csv')

orders = pd.read_csv(
    data_path / 'orders_clean.csv',
    parse_dates=['order_purchase_timestamp']
)

print(df.shape, customers.shape, orders.shape)

(96470, 47) (99441, 5) (99441, 8)


## Section 1: The main table (one row per delivered order)

**What:** Add the real customer id (customer_unique_id) and the city to each order.
**Why:** customer_id is created per ORDER, so only customer_unique_id can show repeat customers.
**Watch out:** a LEFT join with one row per customer_id, so the row count must stay 96,470.

In [4]:
df = df.merge(customers[['customer_id', 'customer_unique_id', 'customer_city']], on='customer_id', how='left')

print(df.shape, ' unique orders:', df['order_id'].nunique())

(96470, 49)  unique orders: 96470


**What:** Create plain date columns (no time) and readable purchase columns.
**Why:** Power BI links tables by date, so the dates must be real dates without hours.
**Watch out:** `.dt.date` keeps only the date part, saved as YYYY-MM-DD.

In [5]:
df['purchase_date'] = df['order_purchase_timestamp'].dt.date
df['delivered_date'] = df['order_delivered_customer_date'].dt.date
df['estimated_date'] = df['order_estimated_delivery_date'].dt.date

df['purchase_year_month'] = df['order_purchase_timestamp'].dt.strftime('%Y-%m')
df['purchase_day_name'] = df['order_purchase_timestamp'].dt.day_name()

df[['purchase_date', 'delivered_date', 'estimated_date', 'purchase_year_month', 'purchase_day_name']].head()

,purchase_date,delivered_date,estimated_date,purchase_year_month,purchase_day_name
0,2017-10-02,2017-10-10,2017-10-18,2017-10,Monday
1,2018-07-24,2018-08-07,2018-08-13,2018-07,Tuesday
2,2018-08-08,2018-08-17,2018-09-04,2018-08,Wednesday
3,2017-11-18,2017-12-02,2017-12-15,2017-11,Saturday
4,2018-02-13,2018-02-16,2018-02-26,2018-02,Tuesday


**What:** Create a readable text column for the target: 'Late' or 'On time'.
**Why:** Charts and slicers look better with words than with 1 and 0 (we keep is_late too, for calculations).
**Watch out:** is_late = 1 means the delivery DATE is after the promised DATE (same definition as SQL, EDA and ML).

In [6]:
df['delivery_status'] = np.where(df['is_late'] == 1, 'Late', 'On time')

print(df['delivery_status'].value_counts())

delivery_status
On time    89936
Late        6534
Name: count, dtype: int64


**What:** Create delay groups (how early or late the order arrived).
**Why:** Shows exactly where customers get unhappy, as in SQL query C6.
**Watch out:** numbers at the start of the labels (1., 2. ...) make Power BI sort them in the right order.

In [7]:
df['delay_bucket'] = pd.cut(df['delay_days'], bins=[-1000, -7, -1, 0, 7, 1000],
                            labels=['1. 7+ days early', '2. 1-6 days early', '3. On the promised date', '4. 1-7 days late', '5. 8+ days late'])

df['delay_bucket'].value_counts().sort_index()

delay_bucket
1. 7+ days early           76140
2. 1-6 days early          12504
3. On the promised date     1292
4. 1-7 days late            3672
5. 8+ days late             2862
Name: count, dtype: int64

**What:** Create distance groups and price groups.
**Why:** Distance and price are numbers; groups are easier to show in charts.
**Watch out:** orders without a distance (486 missing zips) stay empty here, they are NOT dropped.

In [8]:
df['distance_band'] = pd.cut(df['distance_km'], bins=[-1, 100, 300, 600, 1000, 2000, 10000],
                             labels=['1. 0-100 km', '2. 100-300 km', '3. 300-600 km', '4. 600-1000 km', '5. 1000-2000 km', '6. 2000+ km'])

df['price_band'] = pd.cut(df['total_price'], bins=[0, 50, 100, 200, 500, 100000],
                          labels=['1. Under 50', '2. 50-100', '3. 100-200', '4. 200-500', '5. 500+'])

print(df['distance_band'].value_counts().sort_index())
print(df['price_band'].value_counts().sort_index())

distance_band
1. 0-100 km        17904
2. 100-300 km      13336
3. 300-600 km      31491
4. 600-1000 km     17904
5. 1000-2000 km     9743
6. 2000+ km         5606
Name: count, dtype: int64
price_band
1. Under 50    29044
2. 50-100      27606
3. 100-200     25283
4. 200-500     11060
5. 500+         3477
Name: count, dtype: int64


**What:** Set impossible stage times (negative) to empty.
**Why:** A negative number of days is impossible and would distort the averages of the stage charts.
**Watch out:** we only blank the stage time, the order itself stays in the dashboard (real purchase).

In [9]:
print('Negative before:', (df['purchase_to_approval'] < 0).sum(), (df['approval_to_pickup'] < 0).sum(), (df['pickup_to_customer'] < 0).sum())

df['purchase_to_approval'] = df['purchase_to_approval'].where(df['purchase_to_approval'] >= 0)
df['approval_to_pickup'] = df['approval_to_pickup'].where(df['approval_to_pickup'] >= 0)
df['pickup_to_customer'] = df['pickup_to_customer'].where(df['pickup_to_customer'] >= 0)

print('Negative after:', (df['purchase_to_approval'] < 0).sum(), (df['approval_to_pickup'] < 0).sum(), (df['pickup_to_customer'] < 0).sum())

Negative before: 0 1350 23
Negative after: 0 0 0


**What:** Keep only the columns the dashboard needs, in a clean order, and round the long decimals.
**Why:** A smaller file loads faster and the field list in Power BI stays easy to read.
**Watch out:** we keep seller_late_handover here because the dashboard explains the problem (it is NOT used in the ML model).

In [10]:
dashboard_cols = ['order_id', 'customer_unique_id', 'customer_state', 'customer_city', 'seller_id', 'seller_state', 'category',
                  'purchase_date', 'purchase_year_month', 'purchase_day_name', 'purchase_hour',
                  'delivered_date', 'estimated_date', 'delivery_days', 'estimated_days', 'delay_days', 'is_late', 'delivery_status', 'delay_bucket',
                  'purchase_to_approval', 'approval_to_pickup', 'pickup_to_customer', 'seller_late_handover',
                  'items', 'sellers', 'total_price', 'total_freight', 'total_weight_g', 'price_band',
                  'payment_type', 'max_installments', 'total_paid', 'review_score', 'distance_km', 'distance_band', 'same_state']

dashboard_orders = df[dashboard_cols].copy()
dashboard_orders = dashboard_orders.round(2)

print(dashboard_orders.shape)

(96470, 36)


**What:** Save the main table.
**Why:** This is the file you load into Power BI.
**Watch out:** index=False avoids an extra index column.

In [11]:
dashboard_orders.to_csv('dashboard_orders.csv', index=False)
print('Saved dashboard_orders.csv', dashboard_orders.shape)

Saved dashboard_orders.csv (96470, 36)


## Section 2: The table with all orders (only for the cancellation KPI)

**What:** Take every order of any status, with its purchase date and customer state.
**Why:** The main table has only delivered orders, so the cancellation rate and the status mix need all 99,441 orders.
**Watch out:** this table is small on purpose (4 columns).

In [12]:
all_orders = orders.merge(customers[['customer_id', 'customer_state']], on='customer_id', how='left')

all_orders['purchase_date'] = all_orders['order_purchase_timestamp'].dt.date
dashboard_all_orders = all_orders[['order_id', 'order_status', 'purchase_date', 'customer_state']]

print(dashboard_all_orders.shape)
print(dashboard_all_orders['order_status'].value_counts())

(99441, 4)
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


**What:** Save the all-orders table.
**Why:** Needed for the status and cancellation KPIs.
**Watch out:** nothing.

In [13]:
dashboard_all_orders.to_csv('dashboard_all_orders.csv', index=False)
print('Saved dashboard_all_orders.csv')

Saved dashboard_all_orders.csv


## Section 3: The date table

**What:** Build a calendar with one row per day, from the first day of the first month to the last day of the last month.
**Why:** Power BI needs a complete, gapless date table for month-over-month and year-to-date calculations.
**Watch out:** the table must have NO missing days, even days without orders.

In [14]:
first_day = pd.to_datetime(dashboard_all_orders['purchase_date']).min().replace(day=1)
last_day = pd.to_datetime(dashboard_all_orders['purchase_date']).max() + pd.offsets.MonthEnd(0)

dim_date = pd.DataFrame({'date': pd.date_range(first_day, last_day, freq='D')})
dim_date['year'] = dim_date['date'].dt.year
dim_date['month_number'] = dim_date['date'].dt.month
dim_date['month_name'] = dim_date['date'].dt.strftime('%b')
dim_date['year_month'] = dim_date['date'].dt.strftime('%Y-%m')
dim_date['quarter'] = 'Q' + dim_date['date'].dt.quarter.astype(str)
dim_date['weekday_number'] = dim_date['date'].dt.dayofweek + 1
dim_date['weekday_name'] = dim_date['date'].dt.strftime('%a')
dim_date['is_weekend'] = (dim_date['date'].dt.dayofweek >= 5).astype(int)
dim_date['date'] = dim_date['date'].dt.date

print(dim_date.shape, ' from', dim_date['date'].min(), 'to', dim_date['date'].max())
dim_date.head(3)

(791, 9)  from 2016-09-01 to 2018-10-31


,date,year,month_number,month_name,year_month,quarter,weekday_number,weekday_name,is_weekend
0,2016-09-01,2016,9,Sep,2016-09,Q3,4,Thu,0
1,2016-09-02,2016,9,Sep,2016-09,Q3,5,Fri,0
2,2016-09-03,2016,9,Sep,2016-09,Q3,6,Sat,1


**What:** Save the date table.
**Why:** You load it in Power BI and mark it as the date table.
**Watch out:** nothing.

In [15]:
dim_date.to_csv('dim_date.csv', index=False)
print('Saved dim_date.csv')

Saved dim_date.csv


## Section 4: The state table

**What:** Write the 27 Brazilian states with full name and region.
**Why:** The map needs state names, and regions (North, Northeast ...) give a simple way to group the states.
**Watch out:** names have accents, so the file is saved as UTF-8 with BOM so that Power BI reads them correctly.

In [16]:
state_names = {'AC': 'Acre', 'AL': 'Alagoas', 'AM': 'Amazonas', 'AP': 'Amapá', 'BA': 'Bahia', 'CE': 'Ceará', 'DF': 'Distrito Federal',
               'ES': 'Espírito Santo', 'GO': 'Goiás', 'MA': 'Maranhão', 'MG': 'Minas Gerais', 'MS': 'Mato Grosso do Sul', 'MT': 'Mato Grosso',
               'PA': 'Pará', 'PB': 'Paraíba', 'PE': 'Pernambuco', 'PI': 'Piauí', 'PR': 'Paraná', 'RJ': 'Rio de Janeiro', 'RN': 'Rio Grande do Norte',
               'RO': 'Rondônia', 'RR': 'Roraima', 'RS': 'Rio Grande do Sul', 'SC': 'Santa Catarina', 'SE': 'Sergipe', 'SP': 'São Paulo', 'TO': 'Tocantins'}

state_regions = {'AC': 'North', 'AL': 'Northeast', 'AM': 'North', 'AP': 'North', 'BA': 'Northeast', 'CE': 'Northeast', 'DF': 'Central-West',
                 'ES': 'Southeast', 'GO': 'Central-West', 'MA': 'Northeast', 'MG': 'Southeast', 'MS': 'Central-West', 'MT': 'Central-West',
                 'PA': 'North', 'PB': 'Northeast', 'PE': 'Northeast', 'PI': 'Northeast', 'PR': 'South', 'RJ': 'Southeast', 'RN': 'Northeast',
                 'RO': 'North', 'RR': 'North', 'RS': 'South', 'SC': 'South', 'SE': 'Northeast', 'SP': 'Southeast', 'TO': 'North'}

dim_state = pd.DataFrame({'state_code': list(state_names.keys())})
dim_state['state_name'] = dim_state['state_code'].map(state_names)
dim_state['region'] = dim_state['state_code'].map(state_regions)
dim_state['country'] = 'Brazil'

print(dim_state.shape)
dim_state.head()

(27, 4)


,state_code,state_name,region,country
0,AC,Acre,North,Brazil
1,AL,Alagoas,Northeast,Brazil
2,AM,Amazonas,North,Brazil
3,AP,Amapá,North,Brazil
4,BA,Bahia,Northeast,Brazil


**What:** Check that every state in the orders exists in the state table.
**Why:** If a state were missing, its orders would not link to the map in Power BI.
**Watch out:** the printed list must be empty.

In [17]:
states_in_orders = set(dashboard_orders['customer_state'].dropna().unique())
states_in_table = set(dim_state['state_code'])

print('States in orders but not in the state table:', states_in_orders - states_in_table)

States in orders but not in the state table: set()


**What:** Save the state table.
**Why:** You load it in Power BI for the map and the region slicer.
**Watch out:** utf-8-sig keeps the accents correct in Power BI.

In [18]:
dim_state.to_csv('dim_state.csv', index=False, encoding='utf-8-sig')
print('Saved dim_state.csv')

Saved dim_state.csv


## Section 5: Checkpoint numbers (to verify your Power BI dashboard later)
After you build the dashboard, every KPI card must show the same number as below. If a card is different, the DAX formula or the relationship has an error.

**What:** Calculate the main KPIs with pandas.
**Why:** These are your "expected values" for the Power BI cards.
**Watch out:** all numbers come from the saved dashboard tables, the same files Power BI will read.

In [19]:
delivered_orders = len(dashboard_orders)
revenue = dashboard_orders['total_price'].sum()
freight = dashboard_orders['total_freight'].sum()

print('Delivered orders:', delivered_orders)
print('Total orders (all statuses):', len(dashboard_all_orders))
print('Revenue:', round(revenue, 2))
print('Freight:', round(freight, 2), ' Freight % of revenue:', round(freight / revenue * 100, 2))
print('Average order value:', round(revenue / delivered_orders, 2))
print('Average items per order:', round(dashboard_orders['items'].mean(), 3))
print('Active sellers:', dashboard_orders['seller_id'].nunique())
print('Unique customers:', dashboard_orders['customer_unique_id'].nunique())

Delivered orders: 96470
Total orders (all statuses): 99441
Revenue: 13220248.93
Freight: 2198145.9  Freight % of revenue: 16.63
Average order value: 137.04
Average items per order: 1.142
Active sellers: 2960
Unique customers: 93350


**What:** Delivery KPIs: delivery days, promised days, late orders and late %.
**Why:** The core of the project.
**Watch out:** late % must be 6.77%, the same number as in SQL, EDA and ML.

In [20]:
late_orders = dashboard_orders['is_late'].sum()

print('Average delivery days:', round(dashboard_orders['delivery_days'].mean(), 2))
print('Average promised days:', round(dashboard_orders['estimated_days'].mean(), 2))
print('Late orders:', late_orders, ' Late %:', round(late_orders / delivered_orders * 100, 2))
print('Average delay of late orders (days):', round(dashboard_orders[dashboard_orders['is_late'] == 1]['delay_days'].mean(), 2))
print('Severe late (8+ days) %:', round((dashboard_orders['delay_days'] >= 8).mean() * 100, 2))
print('Late seller handover %:', round(dashboard_orders['seller_late_handover'].mean() * 100, 2))
print('Same-state share %:', round(dashboard_orders['same_state'].mean() * 100, 2))
print('Average distance (km):', round(dashboard_orders['distance_km'].mean(), 1))

Average delivery days: 12.56
Average promised days: 23.74
Late orders: 6534  Late %: 6.77
Average delay of late orders (days): 10.62
Severe late (8+ days) %: 2.97
Late seller handover %: 5.78
Same-state share %: 35.97
Average distance (km): 600.4


**What:** Customer experience, money and customer KPIs.
**Why:** Shows the business impact of late deliveries and customer loyalty.
**Watch out:** review KPIs only count orders that have a review.

In [21]:
reviewed = dashboard_orders[dashboard_orders['review_score'].notnull()]
late_revenue = dashboard_orders[dashboard_orders['is_late'] == 1]['total_price'].sum()
orders_per_customer = dashboard_orders.groupby('customer_unique_id')['order_id'].count()

print('Average review score:', round(reviewed['review_score'].mean(), 2))
print('Review score on time / late:', round(reviewed[reviewed['is_late'] == 0]['review_score'].mean(), 2), '/', round(reviewed[reviewed['is_late'] == 1]['review_score'].mean(), 2))
print('1-star %:', round((reviewed['review_score'] == 1).mean() * 100, 2), ' 5-star %:', round((reviewed['review_score'] == 5).mean() * 100, 2))
print('Share of 1-star reviews that come from late orders %:', round(reviewed[(reviewed['review_score'] == 1) & (reviewed['is_late'] == 1)].shape[0] / (reviewed['review_score'] == 1).sum() * 100, 2))
print('Revenue in late orders:', round(late_revenue, 2), ' share %:', round(late_revenue / revenue * 100, 2))
print('Repeat customers:', (orders_per_customer > 1).sum(), ' Repeat customer rate %:', round((orders_per_customer > 1).mean() * 100, 2))
print('Credit card share of orders %:', round((dashboard_orders['payment_type'] == 'credit_card').mean() * 100, 2))
print('Average installments:', round(dashboard_orders['max_installments'].mean(), 2))
print('Cancellation rate % (all orders):', round((dashboard_all_orders['order_status'] == 'canceled').mean() * 100, 2))

Average review score: 4.16
Review score on time / late: 4.29 / 2.27
1-star %: 9.76  5-star %: 59.22
Share of 1-star reviews that come from late orders %: 36.69
Revenue in late orders: 985924.34  share %: 7.46
Repeat customers: 2801  Repeat customer rate %: 3.0
Credit card share of orders %: 76.99
Average installments: 2.93
Cancellation rate % (all orders): 0.63


**What:** Late % by region and the best/worst state.
**Why:** A quick preview of what the map and region chart will show.
**Watch out:** states and regions come from the state table we just built.

In [22]:
region_check = dashboard_orders.merge(dim_state[['state_code', 'region']], left_on='customer_state', right_on='state_code', how='left')
region_late = region_check.groupby('region')['is_late'].agg(['count', 'mean'])
region_late['late_pct'] = (region_late['mean'] * 100).round(2)

region_late[['count', 'late_pct']].sort_values('late_pct', ascending=False)

,count,late_pct
region,,
Northeast,9044,12.72
North,1796,8.57
Central-West,5624,6.53
Southeast,66193,6.12
South,13813,5.90


**What:** List the 4 files you must load into Power BI.
**Why:** Final check that everything was created.
**Watch out:** download the 4 CSV files from the Jupyter folder to your PC.

In [23]:
print('Load these 4 files in Power BI:')
print('1. dashboard_orders.csv      ', dashboard_orders.shape)
print('2. dashboard_all_orders.csv  ', dashboard_all_orders.shape)
print('3. dim_date.csv              ', dim_date.shape)
print('4. dim_state.csv             ', dim_state.shape)

Load these 4 files in Power BI:
1. dashboard_orders.csv       (96470, 36)
2. dashboard_all_orders.csv   (99441, 4)
3. dim_date.csv               (791, 9)
4. dim_state.csv              (27, 4)
